In [17]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/karishmas24/dataset/master_training_data.csv


In [1]:
import os
import time
import joblib
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

In [2]:
# --- KAGGLE FILE PATHS ---
DATA_DIR = '/kaggle/input/datasets/karishmas24/dataset/' 
MODEL_DIR = '/kaggle/working/models'

In [3]:
def objective(trial, X_train_full, y_train_full, scale_weight):
    """Optuna objective function maximizing Average Precision."""
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
    )
    
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400, step=50),
        'max_depth': trial.suggest_int('max_depth', 4, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': scale_weight,
        'tree_method': 'hist',
        'device': 'cuda',
        'random_state': 42,
    }

    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    
    preds = model.predict_proba(X_valid)[:, 1]
    return average_precision_score(y_valid, preds)

In [4]:
def train_business_constrained_xgboost():
    print("=== 💼 BUSINESS MODE: Constrained Risk Optimization ===")
    
    print("\n1. Loading Data...")
    df = pd.read_csv(os.path.join(DATA_DIR, 'master_training_data.csv'))
    df = df.drop(columns=[c for c in ['id', 'card_id', 'merchant_id'] if c in df.columns], errors='ignore')
    
    X = df.drop(columns=['is_fraud', 'client_id']).fillna(-1)
    y = df['is_fraud']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    scale_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
    
    # --- 1. OPTUNA TUNING ---
    print("\n2. Running Optuna (10 Trials for speed)...")
    optuna.logging.set_verbosity(optuna.logging.ERROR) # Silenced for clean output
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, X_train, y_train, scale_weight), n_trials=10)
    
    print("\n3. Training Final Model...")
    best_params = study.best_params
    best_params.update({'scale_pos_weight': scale_weight, 'tree_method': 'hist', 'device': 'cuda', 'random_state': 42})
    
    model = xgb.XGBClassifier(**best_params)
    model.fit(X_train, y_train)

    # --- 2. BUSINESS-CONSTRAINED THRESHOLDING ---
    print("\n4. Calculating Business-Constrained Threshold...")
    y_probs = model.predict_proba(X_test)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)
    
    # Define your exact risk appetite
    MAX_MISSED_FRAUD = 175
    
    # Calculate how many total actual frauds exist in the test set (~2,666)
    total_actual_fraud = np.sum(y_test == 1)
    
    # Mathematically calculate the absolute minimum recall required to hit your target
    target_recall = 1.0 - (MAX_MISSED_FRAUD / total_actual_fraud)
    
    print(f"   -> Target Max Missed Fraud : {MAX_MISSED_FRAUD}")
    print(f"   -> Required System Recall  : {target_recall*100:.2f}%")
    
    # Find the highest possible threshold that still maintains this required recall
    valid_indices = np.where(recalls >= target_recall)[0]
    optimal_threshold = thresholds[valid_indices[-1]]
    
    # --- 3. ROUTING & EVALUATION ---
    passed_mask = y_probs >= optimal_threshold
    final_decisions = np.empty(len(X_test), dtype=object)
    final_decisions[~passed_mask] = "AUTO_APPROVE"
    final_decisions[passed_mask] = "HOLD_FOR_REVIEW"
    
    total_tx = len(X_test)
    auto_approves = np.sum(final_decisions == "AUTO_APPROVE")
    hitl_reviews = np.sum(final_decisions == "HOLD_FOR_REVIEW")
    
    y_test_arr = y_test.values
    false_negatives = np.sum((final_decisions == "AUTO_APPROVE") & (y_test_arr == 1))
    true_positives = np.sum((final_decisions == "HOLD_FOR_REVIEW") & (y_test_arr == 1))
    
    print("\n==================================================")
    print("      CONSTRAINED ROUTING BREAKDOWN (STAGE 1)     ")
    print("==================================================")
    print(f"Total Traffic Processed       : {total_tx:,}")
    print(f"XGBoost Auto-Approved         : {auto_approves:,} ({auto_approves/total_tx*100:.2f}%)")
    print(f"Routed to Review Queue        : {hitl_reviews:,} ({hitl_reviews/total_tx*100:.2f}%)")
    print("==================================================")
    print("             AUTOMATED FILTER ERRORS              ")
    print("==================================================")
    print(f"Actual Fraud Caught           : {true_positives}")
    print(f"Hard False Negatives (Missed) : {false_negatives} (Target: <={MAX_MISSED_FRAUD})")
    print("==================================================\n")

    # ==================================================
    # I ADDED THESE TWO LINES TO FIX YOUR ERROR
    # ==================================================
    os.makedirs(MODEL_DIR, exist_ok=True)
    model.save_model(os.path.join(MODEL_DIR, 'business_xgb_model.json'))
    print(f"✅ Stage 1 Model saved successfully to: {XGB_MODEL_PATH}")

    return optimal_threshold


In [5]:
def train_business_constrained_baselines():
    print("=== 📊 BASELINE MODELS: Logistic Regression & Random Forest ===")

    print("\n1. Loading Data...")
    df = pd.read_csv(os.path.join(DATA_DIR, 'master_training_data.csv'))
    df = df.drop(columns=[c for c in ['id', 'card_id', 'merchant_id'] if c in df.columns], errors='ignore')

    X = df.drop(columns=['is_fraud', 'client_id']).fillna(-1)
    y = df['is_fraud']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # LR needs scaled features; RF/XGB don't
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    MAX_MISSED_FRAUD = 175
    total_actual_fraud = np.sum(y_test == 1)
    target_recall = 1.0 - (MAX_MISSED_FRAUD / total_actual_fraud)

    def evaluate_and_route(model_name, y_probs, model, save_name):
        precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)
        valid_indices = np.where(recalls >= target_recall)[0]
        optimal_threshold = thresholds[valid_indices[-1]] if len(valid_indices) > 0 else 0.5

        passed_mask = y_probs >= optimal_threshold
        final_decisions = np.empty(len(X_test), dtype=object)
        final_decisions[~passed_mask] = "AUTO_APPROVE"
        final_decisions[passed_mask] = "HOLD_FOR_REVIEW"

        total_tx = len(X_test)
        auto_approves = np.sum(final_decisions == "AUTO_APPROVE")
        hitl_reviews = np.sum(final_decisions == "HOLD_FOR_REVIEW")

        y_test_arr = y_test.values
        false_negatives = np.sum((final_decisions == "AUTO_APPROVE") & (y_test_arr == 1))
        true_positives = np.sum((final_decisions == "HOLD_FOR_REVIEW") & (y_test_arr == 1))
        pr_auc = average_precision_score(y_test, y_probs)

        print(f"\n-------------------- {model_name} --------------------")
        print(f"PR-AUC                         : {pr_auc:.4f}")
        print(f"Optimal Threshold              : {optimal_threshold:.4f}")
        print(f"Auto-Approved                  : {auto_approves:,} ({auto_approves/total_tx*100:.2f}%)")
        print(f"Routed to Review Queue         : {hitl_reviews:,} ({hitl_reviews/total_tx*100:.2f}%)")
        print(f"Actual Fraud Caught            : {true_positives}")
        print(f"Hard False Negatives (Missed)  : {false_negatives} (Target: <={MAX_MISSED_FRAUD})")

        os.makedirs(MODEL_DIR, exist_ok=True)
        joblib.dump(model, os.path.join(MODEL_DIR, save_name))
        print(f"✅ {model_name} saved to: {os.path.join(MODEL_DIR, save_name)}")

        return optimal_threshold, pr_auc

    # --- Logistic Regression ---
    print("\n2. Training Logistic Regression...")
    lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    lr_model.fit(X_train_scaled, y_train)
    lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]
    lr_threshold, lr_pr_auc = evaluate_and_route("Logistic Regression", lr_probs, lr_model, 'business_lr_model.pkl')

    # --- Random Forest ---
    print("\n3. Training Random Forest...")
    rf_model = RandomForestClassifier(
        n_estimators=300, max_depth=12, class_weight='balanced',
        n_jobs=-1, random_state=42
    )
    rf_model.fit(X_train, y_train)
    rf_probs = rf_model.predict_proba(X_test)[:, 1]
    rf_threshold, rf_pr_auc = evaluate_and_route("Random Forest", rf_probs, rf_model, 'business_rf_model.pkl')

    print("\n==================================================")
    print("        BASELINE vs XGBOOST — PR-AUC SUMMARY      ")
    print("==================================================")
    print(f"Logistic Regression PR-AUC     : {lr_pr_auc:.4f}")
    print(f"Random Forest PR-AUC           : {rf_pr_auc:.4f}")
    print("(Compare against Stage 1 XGBoost printed above)")
    print("==================================================\n")

    return lr_threshold, rf_threshold


In [6]:
import os
import time
import pandas as pd
import numpy as np
import xgboost as xgb
import tensorflow as tf
import optuna
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Masking, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')

In [7]:
# --- CONFIGURATION ---
DATA_DIR = '/kaggle/input/datasets/karishmas24/dataset/' 
MODEL_DIR = '/kaggle/working/models'
XGB_MODEL_PATH = os.path.join(MODEL_DIR, 'business_xgb_model.json')

In [8]:
# Seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

def optimize_memory(df):
    print("-> Optimizing memory usage...")
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

In [9]:
def build_advanced_stage2_pipeline(STAGE1_THRESHOLD):
    print("=== 🏦 STAGE 2: Advanced Hyperparameter & Threshold Tuning ===")
    
    # ==========================================
    # 1. LOAD DATA & REPRODUCE QUEUE
    # ==========================================
    print("\n1. Loading Data & Recreating Stage 1 Split...")
    df = pd.read_csv(os.path.join(DATA_DIR, 'master_training_data.csv'))
    df = optimize_memory(df)
    
    df['original_idx'] = np.arange(len(df))
    cols_to_drop = [c for c in ['id', 'card_id', 'merchant_id'] if c in df.columns]
    df_stage1 = df.drop(columns=cols_to_drop, errors='ignore')
    
    X = df_stage1.drop(columns=['is_fraud', 'client_id', 'original_idx']).fillna(-1)
    y = df_stage1['is_fraud']
    
    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X, y, df['original_idx'], test_size=0.2, random_state=42, stratify=y
    )
    
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model(XGB_MODEL_PATH)
    preds_stage1 = xgb_model.predict_proba(X_test)[:, 1]
    
    queue_mask = preds_stage1 >= STAGE1_THRESHOLD
    flagged_original_indices = idx_test[queue_mask].values

    # ==========================================
    # 2. BUILD SEQUENCES
    # ==========================================
    print("\n2. Sorting & Building Lookback Sequences (N=5)...")
    df.sort_values(by=['client_id', 'id'], inplace=True)
    df['id_gap_since_last_tx'] = df.groupby('client_id')['id'].diff().fillna(-1)
    df.reset_index(drop=True, inplace=True)
    
    lstm_features = list(X.columns) + ['id_gap_since_last_tx']
    idx_map = {orig: pos for pos, orig in enumerate(df['original_idx'])}
    
    SEQ_LEN = 5
    feature_matrix = df[lstm_features].values
    client_ids = df['client_id'].values
    y_values = df['is_fraud'].values
    
    X_seq, y_seq = [], []
    for orig_idx in flagged_original_indices:
        pos = idx_map[orig_idx]
        target_client = client_ids[pos]
        
        seq = []
        for step in range(SEQ_LEN - 1, -1, -1):
            curr_pos = pos - step
            if curr_pos >= 0 and client_ids[curr_pos] == target_client:
                seq.append(feature_matrix[curr_pos])
            else:
                seq.append(np.zeros(len(lstm_features))) 
        X_seq.append(seq)
        y_seq.append(y_values[pos])
        
    X_seq = np.array(X_seq, dtype=np.float32)
    y_seq = np.array(y_seq, dtype=np.float32)

    # ==========================================
    # 3. SCALE DATA
    # ==========================================
    X_seq_train, X_seq_test, y_seq_train, y_seq_test = train_test_split(
        X_seq, y_seq, test_size=0.2, random_state=42, stratify=y_seq
    )
    
    n_features = X_seq.shape[2]
    scaler = StandardScaler()
    scaler.fit(X_seq_train.reshape(-1, n_features))
    
    def scale_and_preserve_mask(X_tensor):
        flat = X_tensor.reshape(-1, n_features)
        scaled = scaler.transform(flat)
        scaled[np.all(flat == 0, axis=1)] = 0.0
        return scaled.reshape(X_tensor.shape)

    X_seq_train = scale_and_preserve_mask(X_seq_train)
    X_seq_test  = scale_and_preserve_mask(X_seq_test)

    X_t, X_v, y_t, y_v = train_test_split(
        X_seq_train, y_seq_train, test_size=0.2, random_state=42, stratify=y_seq_train
    )

    # ==========================================
    # 4. OPTUNA: LSTM HYPERPARAMETER TUNING
    # ==========================================
    print("\n3. Running Optuna to find best LSTM Architecture (5 Trials)...")
    optuna.logging.set_verbosity(optuna.logging.ERROR)
    
    classes = np.unique(y_t)
    weights = compute_class_weight('balanced', classes=classes, y=y_t)
    class_weights = {classes[i]: weights[i] for i in range(len(classes))}

    def lstm_objective(trial):
        tf.keras.backend.clear_session()
        
        units_1 = trial.suggest_categorical('units_1', [64, 128])
        units_2 = trial.suggest_categorical('units_2', [32, 64])
        dropout_rate = trial.suggest_float('dropout', 0.2, 0.4)
        lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        
        model = Sequential([
            Input(shape=(SEQ_LEN, n_features)),
            Masking(mask_value=0.0), 
            LSTM(units_1, return_sequences=True, use_cudnn=False),
            Dropout(dropout_rate),
            LSTM(units_2, use_cudnn=False),
            Dense(1, activation='sigmoid')
        ])
        
        model.compile(optimizer=Adam(learning_rate=lr), loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='pr_auc', curve='PR')])
        
        early_stop = EarlyStopping(monitor='val_pr_auc', mode='max', patience=3, restore_best_weights=True)
        model.fit(X_t, y_t, validation_data=(X_v, y_v), epochs=15, batch_size=64, class_weight=class_weights, callbacks=[early_stop], verbose=0)
        
        val_loss, val_pr_auc = model.evaluate(X_v, y_v, verbose=0)
        return val_pr_auc

    lstm_study = optuna.create_study(direction='maximize')
    lstm_study.optimize(lstm_objective, n_trials=5)
    
    best_hp = lstm_study.best_params
    print(f"   -> 🏆 Best Architecture: L1={best_hp['units_1']}, L2={best_hp['units_2']}, Drop={best_hp['dropout']:.2f}, LR={best_hp['lr']:.4f}")

    print("\n4. Training Final Optimized LSTM...")
    tf.keras.backend.clear_session()
    final_model = Sequential([
        Input(shape=(SEQ_LEN, n_features)),
        Masking(mask_value=0.0), 
        LSTM(best_hp['units_1'], return_sequences=True, use_cudnn=False),
        Dropout(best_hp['dropout']),
        LSTM(best_hp['units_2'], use_cudnn=False),
        Dense(1, activation='sigmoid')
    ])
    final_model.compile(optimizer=Adam(learning_rate=best_hp['lr']), loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
    
    early_stop = EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True)
    final_model.fit(X_t, y_t, validation_data=(X_v, y_v), epochs=30, batch_size=64, class_weight=class_weights, callbacks=[early_stop], verbose=1)

    # ==========================================
    # 5. OPTUNA: ULTIMATE BUSINESS CONSTRAINT TUNER
    # ==========================================
    print("\n5. Running Ultimate Constraint Optimizer...")
    print("   -> Targets: Queue <= 10k | FN <= 150 | FP <= 1000 | F1 >= 0.70")
    start_time = time.time()
    
    s1_approved_mask = preds_stage1 < STAGE1_THRESHOLD
    s1_auto_approved_count = np.sum(s1_approved_mask)
    s1_missed_fraud = np.sum((s1_approved_mask) & (y_test.values == 1))

    X_seq_scaled_full = scale_and_preserve_mask(X_seq)
    s2_preds = final_model.predict(X_seq_scaled_full, batch_size=256, verbose=0).flatten()
    
    def threshold_objective(trial):
        thresh_approve = trial.suggest_float('thresh_approve', 0.001, 0.30)
        thresh_reject = trial.suggest_float('thresh_reject', 0.30, 0.95)
        
        if thresh_approve >= thresh_reject:
            raise optuna.TrialPruned()
            
        s2_auto_rejected_mask = s2_preds >= thresh_reject
        s2_auto_approved_mask = s2_preds <= thresh_approve
        
        s2_fn = np.sum((s2_auto_approved_mask) & (y_seq == 1))
        s2_fp = np.sum((s2_auto_rejected_mask) & (y_seq == 0))
        hitl_count = np.sum((s2_preds > thresh_approve) & (s2_preds < thresh_reject))
        
        total_fn = s1_missed_fraud + s2_fn
        total_fp = s2_fp
        
        binary_preds = (s2_preds >= thresh_reject).astype(int)
        current_f1 = 0 if np.sum(binary_preds) == 0 else f1_score(y_seq, binary_preds)
        
        cost = 0
        if hitl_count > 10000: cost += (hitl_count - 10000) * 2000 
        if total_fn > 150: cost += (total_fn - 150) * 5000 
        else: cost += total_fn * 10 
        if total_fp > 1000: cost += (total_fp - 1000) * 1000 
        else: cost += total_fp * 1 
        if current_f1 < 0.70: cost += (0.70 - current_f1) * 100000 
            
        return cost

    thresh_study = optuna.create_study(direction='minimize')
    thresh_study.optimize(threshold_objective, n_trials=1000) 
    
    best_approve = thresh_study.best_params['thresh_approve']
    best_reject = thresh_study.best_params['thresh_reject']

    # --- Apply Optimal Thresholds ---
    s2_auto_rejected_mask = s2_preds >= best_reject
    s2_auto_approved_mask = s2_preds <= best_approve
    s2_hitl_mask = (s2_preds > best_approve) & (s2_preds < best_reject)
    
    final_binary_preds = (s2_preds >= best_reject).astype(int)
    current_f1 = f1_score(y_seq, final_binary_preds)

    end_time = time.time()

    # --- Metrics Calculation ---
    total_traffic = len(X_test)
    s2_auto_rejected_count = np.sum(s2_auto_rejected_mask)
    s2_auto_approved_count = np.sum(s2_auto_approved_mask)
    hitl_count = np.sum(s2_hitl_mask)
    
    automation_rate = ((s1_auto_approved_count + s2_auto_approved_count + s2_auto_rejected_count) / total_traffic) * 100
    throughput = total_traffic / (end_time - start_time) if (end_time - start_time) > 0 else 0

    s2_missed_fraud = np.sum((s2_auto_approved_mask) & (y_seq == 1))
    total_hard_fn = s1_missed_fraud + s2_missed_fraud
    total_hard_fp = np.sum((s2_auto_rejected_mask) & (y_seq == 0))

    # --- Confusion Matrix Components ---
    # Breakdown of exactly what happened in the Stage 2 Queue
    tn = np.sum((s2_auto_approved_mask) & (y_seq == 0))      # Good customer, correctly approved
    fn_s2 = np.sum((s2_auto_approved_mask) & (y_seq == 1))   # Fraudster, incorrectly approved
    tp = np.sum((s2_auto_rejected_mask) & (y_seq == 1))      # Fraudster, correctly blocked
    fp = np.sum((s2_auto_rejected_mask) & (y_seq == 0))      # Good customer, incorrectly blocked
    queue_safe = np.sum((s2_hitl_mask) & (y_seq == 0))       # Good customer, sent to human
    queue_fraud = np.sum((s2_hitl_mask) & (y_seq == 1))      # Fraudster, sent to human

    # --- Output ---
    print("\n==================================================")
    print("             FULL SYSTEM BUSINESS METRICS         ")
    print("==================================================")
    print(f"Total Traffic Processed       : {total_traffic:,}\n")
    print(f"Stage 1 Auto-Approved         : {s1_auto_approved_count:,} ({(s1_auto_approved_count/total_traffic)*100:.2f}%)")
    print(f"Stage 2 Auto-Approved (Rescue): {s2_auto_approved_count:,} ({(s2_auto_approved_count/total_traffic)*100:.2f}%)")
    print(f"Stage 2 Auto-Rejected         : {s2_auto_rejected_count:,} ({(s2_auto_rejected_count/total_traffic)*100:.2f}%)")
    print(f"Routed to React Dashboard     : {hitl_count:,} ({(hitl_count/total_traffic)*100:.2f}%)\n")
    
    print(f"Total System Automation Rate  : {automation_rate:.2f}%")
    print(f"Stage 2 Auto-Reject F1 Score  : {current_f1:.4f} (Target: >=0.70)")
    print("==================================================")
    print("             AUTOMATED BLOCKING ERRORS")
    print("                    (PRE-HITL)     ")
    print("==================================================")
    print(f"Hard False Negatives (Missed) : {total_hard_fn:,} (Target: <=150)")
    print(f"Hard False Positives (Blocks) : {total_hard_fp:,} (Target: <=1000)")
    print("==================================================")
    print("             STAGE 2 CONFUSION MATRIX")
    print("          (Where did the Queue end up?)")
    print("==================================================")
    print(f"{'':<20} | {'Auto-Approved':<15} | {'Auto-Rejected':<15} | {'React Dashboard':<15}")
    print("-" * 75)
    print(f"{'Actual Safe (0)':<20} | {tn:<15,} | {fp:<15,} | {queue_safe:<15,}")
    print(f"{'Actual Fraud (1)':<20} | {fn_s2:<15,} | {tp:<15,} | {queue_fraud:<15,}")
    print("==================================================\n")

In [10]:
if __name__ == "__main__":

    calculated_threshold = train_business_constrained_xgboost()
    train_business_constrained_baselines()
    build_advanced_stage2_pipeline(calculated_threshold)

=== 💼 BUSINESS MODE: Constrained Risk Optimization ===

1. Loading Data...

2. Running Optuna (10 Trials for speed)...

3. Training Final Model...

4. Calculating Business-Constrained Threshold...
   -> Target Max Missed Fraud : 175
   -> Required System Recall  : 93.44%

      CONSTRAINED ROUTING BREAKDOWN (STAGE 1)     
Total Traffic Processed       : 1,782,993
XGBoost Auto-Approved         : 1,758,107 (98.60%)
Routed to Review Queue        : 24,886 (1.40%)
             AUTOMATED FILTER ERRORS              
Actual Fraud Caught           : 2491
Hard False Negatives (Missed) : 175 (Target: <=175)

✅ Stage 1 Model saved successfully to: /kaggle/working/models/business_xgb_model.json
=== 📊 BASELINE MODELS: Logistic Regression & Random Forest ===

1. Loading Data...

2. Training Logistic Regression...

-------------------- Logistic Regression --------------------
PR-AUC                         : 0.0176
Optimal Threshold              : 0.2348
Auto-Approved                  : 1,131,000 (63.

I0000 00:00:1782964839.754167      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13736 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782964839.759494      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1782964847.421849     175 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


   -> 🏆 Best Architecture: L1=128, L2=64, Drop=0.40, LR=0.0009

4. Training Final Optimized LSTM...
Epoch 1/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - auc: 0.6909 - loss: 0.6313 - val_auc: 0.7421 - val_loss: 0.6062
Epoch 2/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.7732 - loss: 0.5653 - val_auc: 0.7834 - val_loss: 0.5556
Epoch 3/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.8135 - loss: 0.5229 - val_auc: 0.8128 - val_loss: 0.4990
Epoch 4/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.8430 - loss: 0.4866 - val_auc: 0.8356 - val_loss: 0.4621
Epoch 5/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.8732 - loss: 0.4445 - val_auc: 0.8448 - val_loss: 0.4540
Epoch 6/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.8924 - loss: 0.4121 - val_auc: 0.8469 - val_loss: 0.4255
Epoch 7/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.9109 - loss: 0.3782 - val_auc: 0.8468 - val_loss: 0.4123
Epoch 8/30
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.9262 - loss